# Phase 3 — Embeddings & FAISS

**Goal:** Turn chunks into dense vectors, store them in a FAISS index alongside their metadata, and retrieve the top-k nearest chunks for a query.

By the end you will have:

1. An embedding for every chunk produced in Phase 2.
2. A persisted FAISS index (`data/vector_store/`) you can reload later.
3. A `VectorStore` class with the methods we need for `app/retrieval/vector_store.py`: `add`, `search`, `save`, `load`.
4. A first qualitative test: ask "what is multi-head attention?" and look at what comes back.


## 3.1 — Load chunks from Phase 2


In [ ]:
import os, sys, json
from pathlib import Path
from pydantic import BaseModel
from typing import Optional

repo_root = Path.cwd()
while not (repo_root / "app").is_dir() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
os.chdir(repo_root)
sys.path.insert(0, str(repo_root))

class Chunk(BaseModel):
    chunk_id: str
    doc_id: str
    source: str
    page: int
    text: str
    section: Optional[str] = None
    n_tokens: Optional[int] = None

chunks_path = Path("data/processed/chunks.jsonl")
chunks = [Chunk.model_validate_json(line) for line in chunks_path.read_text().splitlines()]
print(f"Loaded {len(chunks)} chunks")
print("First:", chunks[0].chunk_id, "-", chunks[0].text[:80].replace("\n", " "), "...")


## 3.2 — Pick an embedder

We default to `sentence-transformers/all-MiniLM-L6-v2`:
- 22M parameters, runs on CPU at ~1k sentences/sec on a modern laptop;
- 384-dim output;
- trained on diverse semantic pairs (good general-purpose retrieval).

For higher quality at the cost of speed, swap in `BAAI/bge-large-en-v1.5` (1024-dim, top of the MTEB English leaderboard).


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

EMBEDDER_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBEDDER_NAME)
DIM = embedder.get_sentence_embedding_dimension()
print(f"Embedder    : {EMBEDDER_NAME}")
print(f"Output dim  : {DIM}")
print(f"Max seq len : {embedder.max_seq_length}  tokens")


## 3.3 — Embed all chunks

We normalize embeddings (`normalize_embeddings=True`) so we can use **inner-product** as the FAISS metric, which is mathematically equivalent to cosine similarity for unit-norm vectors.


In [ ]:
import time

texts = [c.text for c in chunks]
t0 = time.time()
embeddings = embedder.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
).astype("float32")
dt = time.time() - t0
print(f"\nEmbedded {len(texts)} chunks in {dt:.1f}s ({len(texts)/dt:.0f}/s)")
print(f"Shape: {embeddings.shape}")
print(f"Norms (should all be ~1.0): min={np.linalg.norm(embeddings, axis=1).min():.3f}, "
      f"max={np.linalg.norm(embeddings, axis=1).max():.3f}")


## 3.4 — Build a FAISS index

For paper-scale corpora (<1M vectors) `IndexFlatIP` is the right choice — exact nearest-neighbor, no training, instant.

For larger corpora you would switch to `IndexHNSWFlat` or `IndexIVFPQ`. We will demo the swap below.


In [ ]:
import faiss

index = faiss.IndexFlatIP(DIM)
index.add(embeddings)
print(f"FAISS index built. ntotal = {index.ntotal}, dim = {index.d}")


## 3.5 — A query, end-to-end


In [ ]:
query = "What is multi-head attention?"
q_vec = embedder.encode([query], normalize_embeddings=True).astype("float32")

scores, indices = index.search(q_vec, k=5)
print(f"Query: {query}\n")
for rank, (i, s) in enumerate(zip(indices[0], scores[0]), start=1):
    c = chunks[i]
    print(f"[{rank}] score={s:.3f}  page={c.page}  chunk={c.chunk_id}")
    print("    ", c.text[:160].replace("\n", " "), "...\n")


## 3.6 — Wrap it all in a reusable `VectorStore`

This is the class that lives in `app/retrieval/vector_store.py`. Two key responsibilities:

1. Maintain a **parallel array** of metadata so we can map FAISS row indices back to chunks.
2. Provide `save` / `load` so we do not re-embed on every restart.


In [ ]:
import faiss, json, numpy as np
from pathlib import Path

class VectorStore:
    def __init__(self, dim: int, embedder: SentenceTransformer):
        self.dim = dim
        self.index = faiss.IndexFlatIP(dim)
        self.metadata: list[dict] = []
        self.embedder = embedder

    def add(self, chunks: list[Chunk]) -> None:
        texts = [c.text for c in chunks]
        embs = self.embedder.encode(
            texts, batch_size=32, normalize_embeddings=True, convert_to_numpy=True
        ).astype("float32")
        self.index.add(embs)
        self.metadata.extend(c.model_dump() for c in chunks)

    def search(self, query: str, k: int = 5) -> list[dict]:
        q = self.embedder.encode([query], normalize_embeddings=True).astype("float32")
        scores, idx = self.index.search(q, k)
        out = []
        for i, s in zip(idx[0], scores[0]):
            if i < 0:
                continue
            m = dict(self.metadata[i])
            m["score"] = float(s)
            out.append(m)
        return out

    def save(self, dir_path: str | Path) -> None:
        p = Path(dir_path); p.mkdir(parents=True, exist_ok=True)
        faiss.write_index(self.index, str(p / "index.faiss"))
        (p / "metadata.jsonl").write_text(
            "\n".join(json.dumps(m) for m in self.metadata), encoding="utf-8"
        )
        (p / "config.json").write_text(json.dumps({"dim": self.dim}), encoding="utf-8")

    @classmethod
    def load(cls, dir_path: str | Path, embedder: SentenceTransformer) -> "VectorStore":
        p = Path(dir_path)
        cfg = json.loads((p / "config.json").read_text())
        store = cls(dim=cfg["dim"], embedder=embedder)
        store.index = faiss.read_index(str(p / "index.faiss"))
        store.metadata = [
            json.loads(line) for line in (p / "metadata.jsonl").read_text().splitlines()
        ]
        return store

# Build, save, reload, re-test.
store = VectorStore(DIM, embedder)
store.add(chunks)
store.save("data/vector_store/aiayn")

reloaded = VectorStore.load("data/vector_store/aiayn", embedder)
hits = reloaded.search("How does scaled dot-product attention work?", k=3)
for h in hits:
    print(f"  page {h['page']}  score={h['score']:.3f}  -- {h['text'][:90]}...")


## 3.7 — Index variants for scale

| Index type           | When to use                                         | Pros                            | Cons                        |
|----------------------|-----------------------------------------------------|---------------------------------|-----------------------------|
| `IndexFlatIP`        | < 1M vectors, exact nearest neighbor                | Simplest, exact, no training    | O(n) per query              |
| `IndexHNSWFlat`      | 1M–10M vectors, fast queries                        | Sub-millisecond queries         | Build time, RAM-heavy       |
| `IndexIVFPQ`         | 10M–100M+, RAM-constrained                          | Compressed storage              | Approximate, needs training |
| Qdrant (server)      | Production: filtering, persistence, multi-tenancy   | Metadata filters, REST API      | Extra service to run        |

We will swap to **Qdrant** in Notebook 09 to show the production path. The `VectorStore` interface above is what makes the swap a one-class change.


In [ ]:
# Demonstrate building an HNSW index (commented out by default — it is slower to build).
# hnsw = faiss.IndexHNSWFlat(DIM, 32)   # 32 = M (graph connectivity)
# hnsw.hnsw.efConstruction = 200
# hnsw.add(embeddings)
# print("HNSW ntotal:", hnsw.ntotal)
print("See Notebook 09 for the Qdrant swap.")


## 3.8 — Sanity checks before moving on

A small retrieval sanity test we will reuse in evaluation:


In [ ]:
sanity_questions = [
    ("multi-head attention", 5),
    ("positional encoding sinusoidal", 5),
    ("encoder decoder stack", 4),
    ("training data WMT 2014", 1),
]

for q, expected_page in sanity_questions:
    hits = reloaded.search(q, k=3)
    pages = [h["page"] for h in hits]
    hit_mark = "ok " if expected_page in pages else "?? "
    print(f"  {hit_mark} '{q}'  -> retrieved pages {pages}  (expected near p.{expected_page})")


## What you learned

- How to embed chunks with `sentence-transformers`.
- Why we normalize embeddings + use inner product as a stand-in for cosine similarity.
- How to build, query, save, and reload a FAISS `IndexFlatIP`.
- The `VectorStore` class shape used in `app/retrieval/vector_store.py`.
- Which FAISS index type to pick at which corpus size.

## Exercises

1. Swap the embedder to `BAAI/bge-small-en-v1.5` (also CPU-friendly) and compare retrieval on the sanity questions.
2. Add a `filter` argument to `VectorStore.search` that only returns chunks from a specific `doc_id` — useful when the user has uploaded multiple papers.
3. Build a tiny eval set of 10 question / expected-page pairs, and measure recall@5.

## Next: [Phase 4 — RAG pipeline](./2026-05-25-phase04-rag-pipeline.ipynb)
